# b3_plan

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): `B3_PLAN`, unfrozen until T4 (FR-D5): the knobs and the candidate rule, the fit cells, folds, loss, grids and seeds, the fit-health rule, the refit, the forecasters and their sign rules, the scoring rule and the common set, Gate B3's counting rule with its minimum of 8, and the experiments it forecasts. Frozen with `coopHash` and pushed before the week-1 fit (NFR-7).

In [ ]:
#| default_exp b3_plan

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.b3_agents import B3_KNOBS
from explRL.b3_cells import CANDIDATES, COARSE, FINE_STEP, FIT_CELLS, FOLDS, REUSED, SIM_SEEDS

B3_PLAN = dict(
    version=1, frozen=False, plan='specs/multiagent/phase22/', exploratory=True,
    hypotheses=dict(
        H22_1='Gate B3, primary: pooled over C1, C2 and C4 part b, the fitted simulator calls more effects right than each of '
              'the unfitted simulator, pilot-only and every-design-helps (the common set, P22-10)',
        H22_2="week-1 check, not a gate: on paper2's 11 conclusive calls, the fitted simulator (fit without the scored study's "
              'cells, refit on its pilot) gets more right than the unfitted 9',
        H22_3="fit check: held-out loss below the unfitted simulator's in at least 2 of the 3 folds S1, S4 and S4b",
        H22_4='descriptive: the refit forecast (fitted) calls more right than the prior-only forecast'),
    knobs=dict(ranges={k: list(v) for k, v in B3_KNOBS.items()}, default="every knob 0; guess the cell's own value",
               trustOne='counting-board agents: a leading value with fewer than q senders counts as known when the cached '
                        'draw for (k, j, value) < trustOne', candidates=list(CANDIDATES),
               choose="chooseKnob: per candidate, the summed held-out loss over S1, S4 and S4b on the coarse grid; the lowest "
                      "wins, ties to 'spread' (P22-2); then the fine grid for that candidate only"),
    fit=dict(cells=[c['id'] for c in FIT_CELLS], targets=dict(clean='pooledShare', liars='contamination'),
             loss='sum over cells of ((sim mean - Claude mean) / Claude se) ** 2, se floored at 0.01 (P22-4)',
             coarse={k: list(v) for k, v in COARSE.items()}, fineStep=dict(FINE_STEP), simSeeds=[SIM_SEEDS[0], SIM_SEEDS[-1]],
             ties='the first grid key in sorted order'),
    folds={k: list(v) for k, v in FOLDS.items()}, reused={k: list(v) for k, v in REUSED.items()},
    fitHealth='stop and ask (P22-11) on a fitted knob at either end of its range in any fit, or on H22-3 failing for the '
              'chosen knob; H22-3 and the retrospective reported for all three candidates, with the caveat that the knob '
              'was chosen on the same folds (P22-18)',
    refit=dict(J='sum over pilot measures ((sim - pilot) / s) ** 2 + sum over refit parameters ((theta - theta0) / tau) ** 2',
               tau='bootTau: SD of the fit over 200 resamples of Claude boards within cells, floor 0.1',
               s='the cached Claude per-board SD of the measure, pooled over its fit cells, over sqrt(pilot boards)',
               params='only parameters with a pilot measure; the others keep theta0 (C1: trustOne on contamination)'),
    forecasters=dict(lines=['fitted', 'priorOnly', 'unfitted', 'pilotOnly', 'naive'], gate=['fitted', 'unfitted', 'pilotOnly', 'naive'],
                     simulators='sign + or - when the 95% bootSeeds interval (2,000 resamples) over seeds 3000-3029 excludes 0, '
                                'else 0', pilotOnly="the sign of the point estimate from the pilot plus the cached rows that "
                                "existed before the experiment; every-design-helps' sign, marked fallback, when an arm is in "
                                'neither (P22-16, P22-17)', naive="the effect's declared naive sign (P22-8)",
                     unrepresented='only the pilot-only and every-design-helps lines; a 0 for both simulators on all items'),
    effects='b3_effects: C1_EFFECTS (8), A2_EFFECTS (4), RETRO_EFFECTS; C2 and C4 part b add theirs in their own frozen plans '
            'before their pilots (P22-15)',
    scoring="paper2's rule: conclusive when the experiment's 95% interval over its boards excludes 0; right when the forecast "
            'has its sign, wrong when it has the other sign or 0, else no call; common set (both simulators represent the '
            'effect) primary, all items secondary',
    gate=dict(hyp='H22_1', rule='on the pooled common set, fitted right > unfitted right, fitted right > pilot-only right and '
                                'fitted right > every-design-helps right', minConclusive=8, below='too few calls to read',
              mcnemar='exact two-sided McNemar p, fitted against each alternative, reported, not gating',
              beside='fitted against pilot-only on the items pilot-only computes (no fallback) (P22-17)',
              refuses='while an experiment in experiments has a frozen forecast but no score (FR-D3)', downstream='none'),
    experiments=dict(simulated=['C1', 'C2', 'C4b'], twoLines=['A2']),
    files=dict(grid='results/multiagent/b3/grid/', fit='results/multiagent/b3/fit.json', retro='results/multiagent/b3/retro.json',
               report='results/multiagent/b3/report.md'),
    runner='tools/b3_run.py (check, probe, grid, fit, retro, forecast <exp>, score <exp>, gate); grid refuses an unfrozen plan')

## Tests

In [ ]:
# test_b3_plan (FR-D5, NFR-7): B3_PLAN holds what FR-D5 lists, is JSON-hashable and unfrozen until T4
from explRL.coop_grid import coopHash
P = B3_PLAN
assert P['frozen'] is False and P['exploratory'] is True and coopHash(P)
assert len(P['fit']['cells']) == 32 and set(P['folds']) == {'S1', 'S4', 'S4b', 'P14'} and P['gate']['minConclusive'] == 8
assert P['knobs']['candidates'] == ['spread', 'shareAll', 'guess'] and P['fit']['simSeeds'] == [3000, 3029]
assert {'H22_1', 'H22_2', 'H22_3', 'H22_4'} == set(P['hypotheses']) and P['experiments']['simulated'] == ['C1', 'C2', 'C4b']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()